# HMM Deployment Workflow Preview

This notebook renders the deployment-only HMM behavioral-state workflow from `behav3d.widgets.state_classification`.

The workflow covers:
- assign HMM intrinsic behavioral states
- combine / rename intrinsic clusters
- rename clusters assigned to binary groups
- create analysis plots
- apply a saved HMM deployment artifact
- open intrinsic or full backprojections


In [1]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from behav3d.widgets.utils import PathPicker
from behav3d.widgets.metadata import MetadataLoader
from behav3d.widgets.state_classification import StateClassificationHMMDeploymentPanel


In [ ]:
output_dir_picker = PathPicker(
    mode="dir",
    start_dir=".",
    description="Output Dir:",
)

metadata_path_picker = PathPicker(
    mode="file",
    start_dir=".",
    description="Metadata CSV:",
)
metadata_path_picker.filter_pattern = "*.csv"

metadata_loader = MetadataLoader(
    metadata_path_picker=metadata_path_picker,
    output_dir_picker=output_dir_picker,
)

render_btn = widgets.Button(
    description="Render HMM Deployment Workflow",
    button_style="success",
)
status_html = widgets.HTML("<i>Load metadata to render the HMM deployment panel.</i>")
panel_out = widgets.Output()


In [ ]:
def _render_hmm_panel(_=None):
    panel_out.clear_output()
    with panel_out:
        if getattr(metadata_loader, "metadata", None) is None:
            status_html.value = "<b style='color:#a60;'>Load metadata first.</b>"
            return
        hmm_deployment_panel = StateClassificationHMMDeploymentPanel(metadata_loader=metadata_loader)
        status_html.value = "<b style='color:#080;'>Rendered HMM deployment panel.</b>"
        display(
            widgets.VBox(
                [
                    widgets.HTML("<h4>HMM Deployment Artifact Workflow</h4>"),
                    hmm_deployment_panel.ui,
                ],
                layout=widgets.Layout(gap="20px"),
            )
        )


_original_load = metadata_loader.load

def _wrapped_load(*args, **kwargs):
    result = _original_load(*args, **kwargs)
    if getattr(metadata_loader, "metadata", None) is not None:
        status_html.value = "<b style='color:#080;'>Metadata loaded. Auto-rendering HMM deployment panel...</b>"
        _render_hmm_panel()
    return result


metadata_loader.load = _wrapped_load
render_btn.on_click(_render_hmm_panel)

display(widgets.VBox([
    widgets.HTML("<h3>Render HMM Deployment Workflow</h3>"),
    output_dir_picker,
    metadata_path_picker,
    metadata_loader.button,
    metadata_loader.out,
    render_btn,
    status_html,
    panel_out,
]))


## Notes

- This panel is the production HMM deployment workflow.
- The preview renders only the HMM deployment workflow.
- Intrinsic and full backprojection buttons are both available in the backprojection step.
